# 05 — Qui-quadrado e V de Cramér corrigido
A causa é avaliada apenas em 2021–2025 por mudança de taxonomia.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
ROOT = Path.cwd()
DATA = ROOT / "data"
OUT = ROOT / "outputs"
for p in [DATA/"raw", DATA/"processed", OUT/"auditoria", OUT/"eda", OUT/"estatistica", OUT/"modelos", OUT/"interpretabilidade", OUT/"modelo_final"]:
    p.mkdir(parents=True,exist_ok=True)
SILVER = DATA/"processed"/"prf_acidentes_silver_2020_2025.parquet"
from scipy.stats import chi2_contingency
df=pd.read_parquet(SILVER)
def analise(col,base):
    tab=pd.crosstab(base[col].astype("string").fillna("Não informado"),base.acidente_grave)
    chi2,p,gl,esperado=chi2_contingency(tab)
    n=tab.values.sum();r,k=tab.shape
    phi2c=max(0,chi2/n-(k-1)*(r-1)/(n-1))
    rc=r-(r-1)**2/(n-1); kc=k-(k-1)**2/(n-1)
    v=np.sqrt(phi2c/min(rc-1,kc-1))
    return dict(variavel=col,cramers_v=v,chi2=chi2,p_valor=p,n_registros=n,pct_esperados_menores_5=100*(esperado<5).mean())
categoricas=["tipo_acidente","uf","br","tipo_pista","fase_dia","hora","condicao_metereologica","dia_semana","uso_solo"]
categoricas += [c for c in df if c.startswith("tracado_") and c!="tracado_via"]
linhas=[analise(c,df) for c in categoricas]
linhas.append(analise("causa_acidente",df[df.ano>=2021]))
ranking=pd.DataFrame(linhas).sort_values("cramers_v",ascending=False)
display(ranking.head(15))
ranking.to_csv(OUT/"estatistica"/"associacoes_cramers_v.csv",index=False)